Import necessary libraries

In [1]:
from langchain_community.utilities import SQLDatabase
from pyprojroot import here
import warnings
warnings.filterwarnings("ignore")

C:\Users\HP\AppData\Local\Temp\ipykernel_15312\3992919677.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Declared the database

In [2]:
# institution_db
institution_db_path = str(here("22_module_assignment/SQLite_DBs")) + "/institutional.db"
institution_db = SQLDatabase.from_uri(f"sqlite:///{institution_db_path}")

# hospitals_db
hospitals_db_path = str(here("22_module_assignment/SQLite_DBs")) + "/hospitals.db"
hospitals_db = SQLDatabase.from_uri(f"sqlite:///{hospitals_db_path}")

# restaurants_db
restaurants_db_path = str(here("22_module_assignment/SQLite_DBs")) + "/restaurants.db"
restaurants_db = SQLDatabase.from_uri(f"sqlite:///{restaurants_db_path}")

Declared LLM (openai/gpt-4.1-mini)

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
#from langchain_community.chat_models import ChatOllama # optonal

# Load variables from .env
load_dotenv()

llm_provider = "openai" # ollama / openai

if llm_provider == "openai":
    github_token = os.getenv("GITHUB_TOKEN")
    endpoint = "https://models.github.ai/inference"
    model_name = "openai/gpt-4.1-mini"

    if not github_token:
        raise ValueError("GITHUB_TOKEN not set in .env")

    llm = ChatOpenAI(
        model_name=model_name,
        openai_api_key=github_token,
        openai_api_base=endpoint,
        temperature=0.5,
    )

import create_sql_agent & tool

In [10]:
from langchain_community.agent_toolkits import create_sql_agent
from langchain_core.tools import tool

institutions_tool

In [23]:
@tool
def institutions_tool(question: str):
    """Answer questions about institutions using the institution database."""
    institutions_agent_executor = create_sql_agent(
        llm,
        db=institution_db,
        agent_type="openai-tools",
        verbose=False
    )

    return institutions_agent_executor.invoke({"input": question})

hospitals_tool

In [24]:
@tool
def hospitals_tool(question: str):
    """Answer questions about hospitals using the hospitals database."""
    hospitals_agent_executor = create_sql_agent(
        llm,
        db=hospitals_db,
        agent_type="openai-tools",
        verbose=False
    )

    return hospitals_agent_executor.invoke({"input": question})

restaurants tool

In [27]:
@tool
def restaurants_tool(question: str):
    """Answer questions about restaurants using the restaurants database."""
    restaurants_agent_executor = create_sql_agent(
        llm,
        db=restaurants_db,
        agent_type="openai-tools",
        verbose=False
    )

    return restaurants_agent_executor.invoke({"input": question})

web search tool

In [34]:
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search_tool(question: str):
    """Answer general knowledge questions using web search (e.g. policies, current events, facts not in our databases)."""

    # 1. Initialize the Tool
    search_tool = DuckDuckGoSearchRun()

    # return the ans
    return search_tool.run(question)

main agent

In [35]:
from langchain.agents import create_agent

# Main Agent
main_agent = create_agent(
    model=llm,
    tools=[
        institutions_tool,
        hospitals_tool,
        restaurants_tool,
        web_search_tool
    ]
)

write your question hear

In [37]:
# TEST 1 USE -> institutions_tool
response = main_agent.invoke({
    "messages": [
        ("user", "give me 3 madrasa name from BARGUNA")
    ]
})

print(response["messages"][-1].content)

Three madrasas in Barguna are:
1. ALHAJ MD. SHAMIM AHSAN DAKHIL MADRASAH, MOHISDANGA
2. AMRAGACHIA SHALEHIA DAKHIL AMDRASAH
3. AMTALI BONDER HOSAINIA FAZIL MADRASHA


In [38]:
# TEST 2 USE -> restaurants_tool
response = main_agent.invoke({
    "messages": [
        ("user", "give me 3 restaurants name inside Dhaka and rating greater than four")
    ]
})

print(response["messages"][-1].content)

Here are 3 restaurants inside Dhaka with ratings greater than four:
1. শুভ এন্টারপ্রাইজ অলটাইম ডিলার, Rating: 5.0, Address: Dhaka - Chittagong Hwy, Minsarai, Bangladesh
2. ভাত ঘর, Rating: 5.0, Address: Dhaka - Chittagong Hwy, Minsarai, Bangladesh
3. আপন নিবাস, Rating: 5.0, Address: Dhaka - Chittagong Hwy, Minsarai, Bangladesh


In [39]:
# TEST 3 USE -> web_search_tool
response = main_agent.invoke({
    "messages": [
        ("user", "What is the healthcare policy of Bangladesh?")
    ]
})

print(response["messages"][-1].content)

The healthcare policy of Bangladesh aims to ensure an effective healthcare system that meets the needs of a healthy nation. The policy provides vision and mission for development, fulfilling the demands of the people while encouraging and inspiring healthcare service providers. The policy covers various thematic areas including legal instruments, financing, coordination and advocacy related to International Health Regulation (IHR), antimicrobial resistance, zoonotic diseases, and food safety.

The government of Bangladesh, along with its development partners, has endorsed a compact recognizing their shared commitment to strengthening the health system and accelerating progress toward Universal Health Coverage. Despite challenges, Bangladesh has made remarkable progress in public health over the past five decades, including increased life expectancy, decreased under-five mortality, and near-universal childhood immunization coverage. However, significant challenges remain in the healthca